In [16]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from kneed import KneeLocator

from kmodes.kprototypes import KPrototypes
from itertools import combinations

from scipy.optimize import linear_sum_assignment

from sklearn.metrics import (
    adjusted_rand_score,
    pairwise_distances,
    silhouette_samples,
    silhouette_score,
)

In [17]:
# Load and clean data 

In [ ]:
# Replace the file name as required
df = pd.read_excel("your_dataset.xlsx")

# Remove leading/trailing whitespace from column names
df.columns = df.columns.str.strip()

In [29]:
# Define clustering variables 

id_col = "ID"
num_cols = ["WWR", "SVR", "AR"]  
cat_cols = ["Height","Wtype", "Stype"]
cluster_col = "cluster"

In [20]:
# data preprocessing 

In [30]:
# Remove rows with missing values in clustering variables

df_clean = df.dropna(subset=num_cols + cat_cols).reset_index(drop=True).copy()

In [31]:
# Ensure numerical variables are numeric

df_clean[num_cols] = df_clean[num_cols].astype(float)

In [33]:
# Standardise categorical variables
for col in cat_cols:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[\\]+", "", regex=True)
        .str.replace(r"\s+", " ", regex=True))

In [24]:
# prepare data for clustering 

In [ ]:
# Select variables used for clustering
selectedcol = df_clean[num_cols + cat_cols].copy()

#Identify categorical column positions required by k-prototypes
categorical_idx = [selectedcol.columns.get_loc(col)
    for col in cat_cols]

print("Dataset size:", selectedcol.shape)
print("Numerical variables:", num_cols)
print("Categorical variables:", cat_cols)
print("Categorical indices:", categorical_idx)

In [26]:
# Selection of number of clusters (k)

In [ ]:
# k selection- Elbow and Kneedle 

# Convert clustering data to NumPy array
X = selectedcol.to_numpy()

# Reference gamma used for k-selection diagnostics. Sensitivity to gamma is evaluated separately.
gamma_ref = 0.5

# Range of cluster solutions to evaluate
k_values = range(2, 11)
costs = []

# Fit k-prototypes for each value of k
for k in k_values:
    kp = KPrototypes(
        n_clusters=k,
        init="Huang",
        n_init=10,
        gamma=gamma_ref,
        random_state=42,
        verbose=0
    )

    kp.fit_predict(X,categorical=categorical_idx)
    costs.append(kp.cost_)


# Identify elbow using Kneedle
kneedle = KneeLocator(
    list(k_values),
    costs,
    curve="convex",
    direction="decreasing"
)

optimal_k = kneedle.elbow

print("Kneedle-selected k:", optimal_k)

# Plot elbow curve
plt.figure(figsize=(7, 5))
plt.plot(k_values, costs, marker="o")

if optimal_k is not None:
    plt.axvline(optimal_k, linestyle="--", label=f"Kneedle elbow: k={optimal_k}")

plt.xlabel("Number of clusters (k)")
plt.ylabel("K-prototypes cost")
plt.title("Elbow Analysis for K-prototypes")
plt.xticks(list(k_values))
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Print cost values
for k, cost in zip(k_values, costs):
    print(f"k={k}: cost={cost:.4f}")

In [ ]:
# Mixed distance 

def mixed_distance_matrix(data, num_cols, cat_cols, gamma):

    X_num = data[num_cols].astype(float).to_numpy()
    X_cat = data[cat_cols].astype(str).to_numpy()

    # Squared Euclidean distance for numerical variables
    D_num = pairwise_distances(X_num, metric="sqeuclidean")

    # Matching dissimilarity for categorical variables
    n = len(data)
    D_cat = np.zeros((n, n))

    for j in range(X_cat.shape[1]):
        D_cat += (X_cat[:, j][:, None] != X_cat[:, j][None, :]).astype(float)

    # Combined mixed distance
    return D_num + gamma * D_cat


def mixed_silhouette(data, labels, num_cols, cat_cols, gamma):
    D_mixed = mixed_distance_matrix(data, num_cols, cat_cols, gamma)
    return silhouette_score(D_mixed, labels, metric="precomputed")


def mixed_silhouette_samples(data, labels, num_cols, cat_cols, gamma):
    D_mixed = mixed_distance_matrix(data, num_cols, cat_cols, gamma)
    return silhouette_samples(D_mixed, labels, metric="precomputed")

In [ ]:
# k selection using mixed distance silhouette 
silhouette_results = []

for k in k_values:

    kp = KPrototypes(
        n_clusters=k,
        init="Huang",
        n_init=10,
        gamma=gamma_ref,
        random_state=42,
        verbose=0
    )

    labels_k = kp.fit_predict(X, categorical=categorical_idx)

    sil = mixed_silhouette(selectedcol, labels_k, num_cols, cat_cols, gamma=gamma_ref)

    silhouette_results.append(sil)

    print(f"k={k}: silhouette={sil:.4f}")
# Plot silhouette coefficients
plt.figure(figsize=(7, 5))
plt.plot(k_values, silhouette_results, marker="o")

plt.xlabel("Number of clusters (k)")
plt.ylabel("Mixed-distance silhouette coefficient")
plt.title("Silhouette Analysis for K-prototypes")
plt.xticks(list(k_values))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# k selection Gap statistic 

B = 100
rng = np.random.default_rng(42)

gap_values = []
gap_se = []

X_real = selectedcol[num_cols + cat_cols].to_numpy()

for k in k_values:

    # Fit k-prototypes to observed data
    kp_real = KPrototypes(
        n_clusters=k,
        init="Huang",
        n_init=10,
        gamma=gamma_ref,
        random_state=42
    )

    kp_real.fit_predict(X_real, categorical=categorical_idx)
    real_log_cost = np.log(kp_real.cost_)

    # Fit k-prototypes to reference datasets
    ref_log_costs = []

    for b in range(B):

        ref = pd.DataFrame(index=selectedcol.index)

        # Generate numerical variables within observed ranges
        for col in num_cols:
            ref[col] = rng.uniform(selectedcol[col].min(), selectedcol[col].max(), len(selectedcol))

        # Generate categorical variables using observed proportions
        for col in cat_cols:
            probs = selectedcol[col].value_counts(normalize=True)
            ref[col] = rng.choice(probs.index, size=len(selectedcol), p=probs.values)

        ref = ref[num_cols + cat_cols]

        kp_ref = KPrototypes(
            n_clusters=k,
            init="Huang",
            n_init=10,
            gamma=gamma_ref,
            random_state=b
        )

        kp_ref.fit_predict(ref.to_numpy(), categorical=categorical_idx)
        ref_log_costs.append(np.log(kp_ref.cost_))

    # Calculate gap statistic and standard error
    gap_values.append(np.mean(ref_log_costs) - real_log_cost)
    gap_se.append(np.std(ref_log_costs, ddof=1) * np.sqrt(1 + 1 / B))


# Select k using the one-standard-error rule
k_list = list(k_values)
best_k = None

for i in range(len(k_list) - 1):
    if gap_values[i] >= gap_values[i + 1] - gap_se[i + 1]:
        best_k = k_list[i]
        break

if best_k is None:
    best_k = k_list[np.argmax(gap_values)]

print("Gap statistic selected k =", best_k)


# Plot gap statistic
plt.figure(figsize=(7, 5))
plt.plot(k_list, gap_values, marker="o")
plt.axvline(best_k, linestyle="--", alpha=0.6, label=f"Selected k: {best_k}")

plt.xlabel("Number of clusters (k)")
plt.ylabel("Gap statistic")
plt.title("Gap Statistic for K-prototypes")
plt.xticks(k_list)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# combine k selection diagnostic 

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))


# (a) Elbow / Kneedle
axes[0].plot(k_values, costs, marker="o")
axes[0].axvline(optimal_k, linestyle="--", alpha=0.7, label=f"Selected k = {optimal_k}")

axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("K-prototypes cost")
axes[0].set_title("(a) Elbow/Kneedle analysis")
axes[0].set_xticks(list(k_values))
axes[0].legend()


# (b) Mixed-distance silhouette
best_k_sil = list(k_values)[np.argmax(silhouette_results)]

axes[1].plot(k_values, silhouette_results, marker="o")
axes[1].axvline(best_k_sil, linestyle="--", alpha=0.7, label=f"Selected k = {best_k_sil}")

axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Mixed-distance silhouette coefficient")
axes[1].set_title("(b) Silhouette analysis")
axes[1].set_xticks(list(k_values))
axes[1].legend()


# (c) Gap-type reference analysis
axes[2].plot(k_values, gap_values, marker="o")
axes[2].axvline(best_k, linestyle="--", alpha=0.7, label=f"Selected k = {best_k}")

axes[2].set_xlabel("Number of clusters (k)")
axes[2].set_ylabel("Gap statistic")
axes[2].set_title("(c) Gap-type reference analysis")
axes[2].set_xticks(list(k_values))
axes[2].legend()


# Save and display figure
plt.tight_layout()
plt.savefig("cluster_number_selection.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# Grid search- parameter sensitivity and stability 

In [ ]:
# initial parameter screening 

k_grid = [2, 3, 4]
gamma_grid = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
init_grid = ["Huang"]

n_init = 10
random_state = 42

grid_results = []
grid_labels = {}

X = selectedcol.to_numpy()


# Run grid search
for k in k_grid:
    for gamma in gamma_grid:
        for init_method in init_grid:

            kp = KPrototypes(
                n_clusters=k,
                init=init_method,
                n_init=n_init,
                gamma=gamma,
                random_state=random_state,
                verbose=0
            )

            labels_grid = kp.fit_predict(X, categorical=categorical_idx)

            # Mixed-distance silhouette coefficient
            sil = mixed_silhouette(selectedcol, labels_grid, num_cols, cat_cols, gamma)

            # Cluster sizes
            cluster_sizes = pd.Series(labels_grid).value_counts()

            # Store results
            grid_results.append({
                "k": k,
                "gamma": gamma,
                "init": init_method,
                "cost": kp.cost_,
                "silhouette": sil,
                "min_cluster_size": cluster_sizes.min(),
                "max_cluster_size": cluster_sizes.max()
            })

            # Store cluster assignments
            grid_labels[(k, gamma, init_method)] = labels_grid.copy()


# Convert results to DataFrame
grid_results = pd.DataFrame(grid_results)

print("Number of configurations:", len(grid_results))
grid_results

In [ ]:
# parameter sensitivity for k=4

k_fixed = 4

gamma_grid = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
init_grid = ["Huang", "Cao", "random"]
n_init_grid = [10, 30, 50]
seed_grid = [0, 10, 20, 42, 99]

k4_grid_results = []
k4_grid_labels = {}

X = selectedcol.to_numpy()


for gamma in gamma_grid:
    for init_method in init_grid:
        for n_init_test in n_init_grid:
            for seed in seed_grid:

                kp = KPrototypes(
                    n_clusters=k_fixed,
                    init=init_method,
                    n_init=n_init_test,
                    gamma=gamma,
                    random_state=seed,
                    verbose=0
                )

                labels = kp.fit_predict(X, categorical=categorical_idx)

                # Mixed-distance silhouette coefficient
                sil = mixed_silhouette(selectedcol, labels, num_cols, cat_cols, gamma)

                # Cluster balance
                cluster_sizes = pd.Series(labels).value_counts()
                min_size = cluster_sizes.min()
                max_size = cluster_sizes.max()
                balance_ratio = min_size / max_size

                # Store results
                k4_grid_results.append({
                    "gamma": gamma,
                    "init": init_method,
                    "n_init": n_init_test,
                    "seed": seed,
                    "cost": kp.cost_,
                    "silhouette": sil,
                    "min_cluster_size": min_size,
                    "max_cluster_size": max_size,
                    "balance_ratio": balance_ratio
                })

                # Store cluster assignments for stability analysis
                k4_grid_labels[(gamma, init_method, n_init_test, seed)] = labels.copy()


# Convert results to DataFrame
k4_grid_results = pd.DataFrame(k4_grid_results)

print("Number of configurations:", len(k4_grid_results))
k4_grid_results.head()

In [ ]:
# performance and stability across gamma

gamma_summary = (
    k4_grid_results
    .groupby("gamma")
    .agg(
        best_cost=("cost", "min"),
        mean_cost=("cost", "mean"),
        sd_cost=("cost", "std"),
        best_silhouette=("silhouette", "max"),
        mean_silhouette=("silhouette", "mean"),
        sd_silhouette=("silhouette", "std"),
        mean_balance_ratio=("balance_ratio", "mean"),
        sd_balance_ratio=("balance_ratio", "std")
    )
    .reset_index()
)
gamma_stability = []

for gamma in gamma_grid:

    gamma_keys = [key for key in k4_grid_labels if key[0] == gamma]
    ari_values = []

    for key1, key2 in combinations(gamma_keys, 2):
        labels1 = k4_grid_labels[key1]
        labels2 = k4_grid_labels[key2]

        ari_values.append(adjusted_rand_score(labels1, labels2))

    gamma_stability.append({
        "gamma": gamma,
        "mean_ARI": np.mean(ari_values),
        "sd_ARI": np.std(ari_values, ddof=1),
        "min_ARI": np.min(ari_values)
    })

gamma_stability = pd.DataFrame(gamma_stability)

# gamma summary 

gamma_comparison = gamma_summary.merge(gamma_stability, on="gamma")

gamma_comparison.round(3)

In [ ]:
# Algorithmic sensitivity at selected gamma
selected_gamma = 0.5

selected_gamma_results = k4_grid_results[
    k4_grid_results["gamma"] == selected_gamma
].copy()

# Summary by initialisation method
init_summary = (
    selected_gamma_results
    .groupby("init")
    .agg(
        mean_cost=("cost", "mean"),
        sd_cost=("cost", "std"),
        mean_silhouette=("silhouette", "mean"),
        sd_silhouette=("silhouette", "std"),
        mean_balance_ratio=("balance_ratio", "mean"),
        sd_balance_ratio=("balance_ratio", "std")
    )
    .reset_index()
)

# Summary by number of initialisations
ninit_summary = (
    selected_gamma_results
    .groupby("n_init")
    .agg(
        mean_cost=("cost", "mean"),
        sd_cost=("cost", "std"),
        mean_silhouette=("silhouette", "mean"),
        sd_silhouette=("silhouette", "std"),
        mean_balance_ratio=("balance_ratio", "mean"),
        sd_balance_ratio=("balance_ratio", "std")
    )
    .reset_index()
)


# Summary by random seed
seed_summary = (
    selected_gamma_results
    .groupby("seed")
    .agg(
        mean_cost=("cost", "mean"),
        sd_cost=("cost", "std"),
        mean_silhouette=("silhouette", "mean"),
        sd_silhouette=("silhouette", "std"),
        mean_balance_ratio=("balance_ratio", "mean"),
        sd_balance_ratio=("balance_ratio", "std")
    )
    .reset_index()
)


In [ ]:
# Calculate assignment stability 

def parameter_stability(parameter, values, key_position):

    results = []

    for value in values:
        keys = [
            key for key in k4_grid_labels
            if key[0] == selected_gamma and key[key_position] == value
        ]

        ari_values = [
            adjusted_rand_score(k4_grid_labels[key1], k4_grid_labels[key2])
            for key1, key2 in combinations(keys, 2)
        ]

        results.append({
            parameter: value,
            "mean_ARI": np.mean(ari_values),
            "sd_ARI": np.std(ari_values, ddof=1),
            "min_ARI": np.min(ari_values)
        })

    return pd.DataFrame(results)

In [ ]:
# Calculate stability
init_stability = parameter_stability("init", init_grid, 1)
ninit_stability = parameter_stability("n_init", n_init_grid, 2)
seed_stability = parameter_stability("seed", seed_grid, 3)

# Combine performance and stability results
init_comparison = init_summary.merge(init_stability, on="init")
ninit_comparison = ninit_summary.merge(ninit_stability, on="n_init")
seed_comparison = seed_summary.merge(seed_stability, on="seed")

# Display results
display(init_comparison.round(3))
display(ninit_comparison.round(3))
display(seed_comparison.round(3))

In [ ]:
# Final model and cluster assignments 

In [ ]:
gamma_final = 0.5

kp_final = KPrototypes(
    n_clusters=4,
    init="Huang",
    n_init=30,
    gamma=gamma_final,
    random_state=42,
    verbose=0
)

labels_final = kp_final.fit_predict(X, categorical=categorical_idx)

# Add cluster assignments to the cleaned dataset
df_clustered = df_clean.copy()
df_clustered["Cluster"] = labels_final

# Cluster size and balance
cluster_sizes = pd.Series(labels_final).value_counts().sort_index()
balance_ratio = cluster_sizes.min() / cluster_sizes.max()

print("Cluster sizes:")
print(cluster_sizes)
print("\nBalance ratio:", round(balance_ratio, 3))


In [ ]:
# Cluster profile summary 

# Mean values of numerical variables
numeric_summary = (
    df_clustered
    .groupby("Cluster")[num_cols]
    .mean()
    .round(3)
)

# Modal categories
categorical_summary = (
    df_clustered
    .groupby("Cluster")[cat_cols]
    .agg(lambda x: x.mode().iloc[0])
)

# Combine numerical and categorical summaries
cluster_summary = numeric_summary.join(categorical_summary)

# Add cluster size
cluster_summary["n"] = df_clustered.groupby("Cluster").size()

cluster_summary

In [ ]:
# cluster evaluation 

In [ ]:

# Overall mixed-distance silhouette coefficient
overall_silhouette = mixed_silhouette(
    selectedcol, labels_final, num_cols, cat_cols, gamma_final
)

print("Overall mixed silhouette:", round(overall_silhouette, 3))


# Observation-level silhouette coefficients
silhouette_values = mixed_silhouette_samples(
    selectedcol, labels_final, num_cols, cat_cols, gamma_final
)

df_clustered["Silhouette"] = silhouette_values

# Mean silhouette coefficient by cluster
cluster_silhouette = (
    df_clustered
    .groupby("Cluster")["Silhouette"]
    .mean()
    .round(3)
)

print("\nMean silhouette by cluster:")
print(cluster_silhouette)


In [ ]:
# within cluster compactness 

# Extract final cluster prototypes
centroids = kp_final.cluster_centroids_

# Handle different kmodes output formats
if isinstance(centroids, (tuple, list)) and len(centroids) == 2:
    cent_num = np.asarray(centroids[0], dtype=float)
    cent_cat = np.asarray(centroids[1], dtype=object)
else:
    centroids = np.asarray(centroids, dtype=object)
    cent_num = centroids[:, :len(num_cols)].astype(float)
    cent_cat = centroids[:, len(num_cols):].astype(object)

# Numerical and categorical data
X_num = selectedcol[num_cols].to_numpy(dtype=float)
X_cat = selectedcol[cat_cols].astype(str).to_numpy()

# Calculate mixed dissimilarity from assigned prototype
distance_to_prototype = []

for i in range(len(selectedcol)):
    c = labels_final[i]

    numeric_distance = np.sum((X_num[i] - cent_num[c]) ** 2)
    categorical_distance = np.sum(X_cat[i] != cent_cat[c].astype(str))

    total_distance = numeric_distance + gamma_final * categorical_distance
    distance_to_prototype.append(total_distance)

# Add distances to clustered dataset
df_clustered["Distance_to_prototype"] = distance_to_prototype

# Summarise within-cluster compactness
compactness_summary = (
    df_clustered
    .groupby("Cluster")["Distance_to_prototype"]
    .agg(
        cluster_size="count",
        mean_distance="mean",
        median_distance="median",
        sd_distance="std",
        min_distance="min",
        max_distance="max"
    )
    .round(4)
)

compactness_summary

In [ ]:
# cluster characterisation 
# Numerical summary: mean and standard deviation
numeric_summary = (
    df_clustered
    .groupby("Cluster")[num_cols]
    .agg(["mean", "std"])
    .round(3)
)

# Categorical summary: dominant category (mode)
categorical_summary = (
    df_clustered
    .groupby("Cluster")[cat_cols]
    .agg(lambda x: x.mode().iloc[0])
)

display(numeric_summary)
display(categorical_summary)


In [ ]:
# observation closest to cluster prototype 

closest_to_prototype = (
    df_clustered
    .sort_values(["Cluster", "Distance_to_prototype"])
    .groupby("Cluster")
    .head(5)
    [["ID", "Cluster", "Distance_to_prototype"] + num_cols + cat_cols]
    .reset_index(drop=True)
)

closest_to_prototype

In [ ]:
# Subsampling stability analysis 

In [ ]:
retention_levels = [0.90, 0.80, 0.70, 0.60]
n_repetitions = 100

overall_results = []
profile_results = []

for retention in retention_levels:

    sample_n = round(len(selectedcol) * retention)

    # Separate reproducible random generator for each retention level
    rng = np.random.default_rng(2026)

    for rep in range(n_repetitions):

        # Random subsample without replacement
        sample_idx = np.sort(
            rng.choice(len(selectedcol), size=sample_n, replace=False)
        )

        data_sub = selectedcol.iloc[sample_idx].copy()

        # Re-run K-prototypes using final parameters
        kp_sub = KPrototypes(
            n_clusters=4,
            init="Huang",
            n_init=30,
            gamma=gamma_final,
            random_state=42,
            verbose=0
        )

        labels_sub = np.asarray(
            kp_sub.fit_predict(data_sub.to_numpy(), categorical=categorical_idx)
        )

        # Full-sample labels for the retained observations
        original_sub_labels = np.asarray(labels_final)[sample_idx]

        # Membership stability
        ari = adjusted_rand_score(original_sub_labels, labels_sub)

        # Match subsample labels to full-sample cluster labels
        contingency = pd.crosstab(original_sub_labels, labels_sub)
        contingency = contingency.reindex(index=range(4), columns=range(4), fill_value=0)

        row_ind, col_ind = linear_sum_assignment(-contingency.to_numpy())
        cluster_mapping = {sub: full for full, sub in zip(row_ind, col_ind)}
        matched_labels = np.array([cluster_mapping[x] for x in labels_sub])

        # Store overall result
        overall_results.append({
            "Retention": retention,
            "Repetition": rep + 1,
            "ARI": ari
        })

        # Store matched cluster profiles
        df_sub = data_sub.copy()
        df_sub["Cluster"] = matched_labels

        for cluster in range(4):
            cluster_data = df_sub[df_sub["Cluster"] == cluster]

            profile_results.append({
                "Retention": retention,
                "Repetition": rep + 1,
                "Cluster": cluster,
                "n": len(cluster_data),
                "WWR": cluster_data["WWR"].mean(),
                "SVR": cluster_data["SVR"].mean(),
                "AR": cluster_data["AR"].mean(),
                "Height": cluster_data["Height"].mode().iloc[0],
                "Wtype": cluster_data["Wtype"].mode().iloc[0],
                "Stype": cluster_data["Stype"].mode().iloc[0]
            })

overall_results = pd.DataFrame(overall_results)
profile_results = pd.DataFrame(profile_results)

In [ ]:
# Membership stability by retention level
ari_summary = (
    overall_results
    .groupby("Retention")["ARI"]
    .agg(["mean", "median", "std", "min", "max"])
    .round(3)
)

ari_summary

In [ ]:
# Mean numerical cluster profiles across repetitions
numeric_profile_summary = (
    profile_results
    .groupby(["Retention", "Cluster"])[num_cols + ["n"]]
    .mean()
    .round(3)
)

numeric_profile_summary

In [ ]:
# Full-sample categorical modes
full_modes = (
    df_clustered
    .groupby("Cluster")[cat_cols]
    .agg(lambda x: x.mode().iloc[0])
)

# Frequency of retaining the full-sample categorical mode
selection_frequency = []

for retention in retention_levels:
    for cluster in range(4):

        cluster_profiles = profile_results[
            (profile_results["Retention"] == retention) &
            (profile_results["Cluster"] == cluster)
        ]

        row = {"Retention": retention, "Cluster": cluster}

        for var in cat_cols:
            full_mode = full_modes.loc[cluster, var]
            row[f"{var}_frequency"] = (cluster_profiles[var] == full_mode).mean() * 100

        selection_frequency.append(row)

selection_frequency = pd.DataFrame(selection_frequency).round(1)

selection_frequency

In [ ]:
# numeric profile deviation 

# Full-sample numerical profiles
full_numeric = (
    df_clustered
    .groupby("Cluster")[num_cols]
    .mean()
    .round(3)
)

# Subsample numerical profiles
subsample_numeric = (
    profile_results
    .groupby(["Retention", "Cluster"])[num_cols]
    .mean()
    .round(3)
)

# Difference from full-sample profile
numeric_deviation = (
    subsample_numeric
    .sub(full_numeric, level="Cluster")
    .round(3)
)

numeric_deviation

In [ ]:
# cluster number stability under subsampling 

In [ ]:
retention_levels = [0.90, 0.80, 0.70, 0.60]
k_values = range(2, 11)
n_repetitions = 100

elbow_results = []
failed_fits = []

for retention in retention_levels:

    sample_n = round(len(selectedcol) * retention)
    rng = np.random.default_rng(2026)

    for rep in range(n_repetitions):

        sample_idx = np.sort(rng.choice(len(selectedcol), size=sample_n, replace=False))
        data_sub = selectedcol.iloc[sample_idx].copy()
        X_sub = data_sub.to_numpy()

        for k in k_values:

            try:
                kp = KPrototypes(
                    n_clusters=k,
                    init="Huang",
                    n_init=10,
                    gamma=gamma_ref,
                    random_state=42,
                    verbose=0
                )

                kp.fit_predict(X_sub, categorical=categorical_idx)

                elbow_results.append({
                    "Retention": retention,
                    "Repetition": rep + 1,
                    "k": k,
                    "Cost": kp.cost_
                })

            except ValueError:
                failed_fits.append({
                    "Retention": retention,
                    "Repetition": rep + 1,
                    "k": k
                })

elbow_results = pd.DataFrame(elbow_results)
failed_fits = pd.DataFrame(failed_fits)

print("Successful fits:", len(elbow_results))
print("Failed fits:", len(failed_fits))

In [ ]:
# Mean cost at each k and retention level
mean_costs = (
    elbow_results
    .groupby(["Retention", "k"])["Cost"]
    .mean()
    .reset_index()
)

In [ ]:
# Apply Kneedle to mean cost curves
kneedle_results = []

for retention in retention_levels:

    data = mean_costs[mean_costs["Retention"] == retention]

    kneedle = KneeLocator(
        data["k"],
        data["Cost"],
        curve="convex",
        direction="decreasing"
    )

    kneedle_results.append({
        "Retention": retention,
        "Selected_k": kneedle.elbow
    })

kneedle_results = pd.DataFrame(kneedle_results)

kneedle_results